# Лабораторная работа 11. LSTM для временных рядов

**Цель.** Подготовить последовательности для рекуррентной нейросети и обучить LSTM-прогноз по временным рядам ЭКГ.


## Ход работы

Главная часть здесь — подготовка данных к нейросети. Для LSTM временной ряд нужно превратить в набор коротких окон, где по нескольким предыдущим точкам предсказывается следующая.


## Подготовка структуры данных

Задаю путь к папке с рядами и подключаю библиотеки. Дальше все найденные сигналы должны быть приведены к одному формату окон.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
data_root = Path('data/underwork')


## Оконное представление

Функция `make_windows` нарезает ряд на фрагменты фиксированной длины. Такое представление удобно для LSTM: модель получает последовательность наблюдений, а целевым значением становится следующая точка.


In [ ]:
def make_windows(values, window_size=24):
    values = np.asarray(values, dtype=np.float32)
    x, y = [], []
    for i in range(len(values) - window_size):
        x.append(values[i:i + window_size])
        y.append(values[i + window_size])
    return np.asarray(x), np.asarray(y)

if not data_root.exists():
    print('Папка с ЭКГ-данными не найдена, блок подготовки датасета пропущен.')
else:
    frames = []
    for calm_path in sorted(data_root.glob('*/calm_p.csv')):
        df = pd.read_csv(calm_path)
        if '1' in df.columns:
            frames.append(df['1'].rename(calm_path.parent.name))
    print(f'Найдено рядов: {len(frames)}')
    if frames:
        values = frames[0].dropna().to_numpy()
        x, y = make_windows(values)
        print('Размер X:', x.shape, 'Размер y:', y.shape)
        plt.figure(figsize=(10, 3))
        plt.plot(values[:1000])
        plt.title('Первый найденный ряд')
        plt.show()


## Вывод

Для LSTM важнее всего правильно подготовить вход: ряд превращается в пары «окно — следующее значение». После этого тот же принцип можно использовать для обучения рекуррентной модели.
